# Implementation Colour Rejection Strategy

This notebook implements a color-based rejection strategy to improve the quality of annotated keypoints in newborn thermal and aligned RGB image datasets.

## Main Steps

1. **Load Annotations and Images**
   - Loads a JSON file containing keypoint annotations and metadata for each image pair.
   - Sets up paths for aligned RGB images and grayscale thermal images.

2. **Extract Baby ID**
   - Defines a function to extract the baby ID from the image filename for grouping and processing.

3. **Color Extraction**
   - For each annotated point with a valid temperature, extracts a small region around the keypoint in the aligned RGB image (converted to HSV).
   - Filters pixels within a predefined skin color range.
   - Computes the average color of the filtered region to represent the expected skin color for that baby.

4. **Annotation Filtering**
   - For each keypoint, compares its color to the reference color.
   - If the color difference exceeds specified thresholds (for hue, saturation, and value), the annotation is rejected (temperature set to `None`).
   - Keeps track of removed annotations for further analysis.

5. **Save Cleaned Annotations**
   - Writes the cleaned annotation data (with rejected points removed) to a new JSON file.

6. **Visualization**
   - Optionally visualizes removed keypoints for a specific baby, drawing circles on the corresponding RGB images.

7. **Additional Utilities (Commented)**
   - Code snippets for identifying images dominated by blue tones.
   - Outlier detection and visualization for further dataset cleaning.

## Purpose

This strategy helps to automatically reject keypoints whose color characteristics do not match the expected skin color, reducing annotation errors due to occlusions, artifacts, or mislabeling. The process is performed per baby to account for individual color variations.



In [1]:
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

BASE_PATH=r"D:\newbornAligned"

BASE_PATH_GRAYSCALE=r"D:\newbornGrayscale"

with open("json/final_dataset_annotated_zone_v2.json", "r") as file:
    data=json.load(file)

print(len(data))

106450


In [4]:
def extract_baby_id(dic):
    return int(dic["thermal_image"][:2])

def color_extract(dic):
    path=os.path.join(BASE_PATH, dic["aligned_image"])

    image=cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2HSV)
    SIDE=2
    LOWER_SKIN=np.array([0, 40, 20], dtype=np.uint8)
    UPPER_SKIN=np.array([25, 255, 242], dtype=np.uint8)

    x_shape, y_shape, _=image.shape
    region=[]
    for point in dic["anotaciones"]:
        if(point["temperature"]!=None):
            x_start=max(point["y"] - SIDE, 0)
            x_end=min(point["y"] + SIDE + 1, x_shape)
            y_start=max(point["x"] - SIDE, 0)
            y_end=min(point["x"] + SIDE + 1, y_shape)
            for i in range(x_start, x_end):
                for j in range(y_start, y_end):
                    region.append(image[i, j])

    pixels=np.array(region)
    h, s, v=pixels[:, 0], pixels[:, 1], pixels[:, 2]

    mask=((h>=LOWER_SKIN[0]) & (h<=UPPER_SKIN[0]) & (s>=LOWER_SKIN[1]) & (s<=UPPER_SKIN[1]) & (v>=LOWER_SKIN[2]) & (v<=UPPER_SKIN[2]))

    filtered=pixels[mask]
    
    if(list(filtered)):
        avg_color=np.mean(filtered, axis=0)
    else:
        return None

    avg_color=avg_color.astype(int)
    return avg_color.tolist()

def filter_annotations(dic, color, thres_h=15, thres_s=60, thres_v=90):
    image=cv2.cvtColor(cv2.imread(os.path.join(BASE_PATH, dic["aligned_image"])), cv2.COLOR_BGR2HSV)
    image=np.array(image)
    grayscale_image=cv2.imread(os.path.join(BASE_PATH_GRAYSCALE, dic["thermal_image"]), cv2.IMREAD_GRAYSCALE)
    x_shape, y_shape, _=image.shape
    NEW_SIDE=1

    removed=0
    list_removed=[]

    for i in range(len(dic["anotaciones"])):
        if(dic["anotaciones"][i]["temperature"]!=None):
            region=[]

            x_start=max(dic["anotaciones"][i]["y"] - NEW_SIDE, 0)
            x_end=min(dic["anotaciones"][i]["y"] + NEW_SIDE + 1, x_shape)
            y_start=max(dic["anotaciones"][i]["x"] - NEW_SIDE, 0)
            y_end=min(dic["anotaciones"][i]["x"] + NEW_SIDE + 1, y_shape)

            for a in range(x_start, x_end):
                for b in range(y_start, y_end):
                    region.append(grayscale_image[a, b])

            region=np.array(region)
            index=np.argmax(region)

            true_x, true_y=x_start + index % (x_end-x_start), y_start + index // (x_end-x_start)

            point_color=image[true_x, true_y]

            diff=point_color-color
            diff[0], diff[1], diff[2]=min(abs(diff[0]), 180-abs(diff[0])), abs(diff[1]), abs(diff[2])

            if (diff[0]>thres_h or diff[1]>thres_s or diff[2]>thres_v):
                removed+=1
                list_removed.append([dic, dic["anotaciones"][i]["class"]])
                dic["anotaciones"][i]["temperature"]=None

    return removed, list_removed, dic

In [ ]:
removed=0
list_removed=[]
l_diff=[]
l_diff_abs=[]
data_clean=[]
previous_baby_id=0 
color=None
for i in range(len(data)): 
    baby_id=extract_baby_id(data[i])

    if(baby_id!=previous_baby_id):
        print(baby_id)
        previous_baby_id=baby_id
        color=None

    new_color=color_extract(data[i])
    if new_color:
        color=new_color

    if color:
        removed_temp, list_removed_temp, dic=filter_annotations(data[i], color)

    removed+=removed_temp
    list_removed.extend(list_removed_temp)
    data_clean.append(dic)

In [3]:
with open("json/annotations_final_v2.json", "w") as file:
    json.dump(data_clean, file, indent=4)

In [2]:
with open("json/list_removed_2.json", "r") as file:
    list_removed=json.load(file)

In [8]:
print(len(list_removed))

19253


In [ ]:
i=0
baby=38
for dic, point in list_removed:
    image=cv2.cvtColor(cv2.imread(os.path.join(BASE_PATH, dic["aligned_image"])), cv2.COLOR_BGR2RGB)
    if(extract_baby_id(dic)==baby):
        for p in dic["anotaciones"]:
            if p["class"]==point:
                cv2.circle(image, (p["x"], p["y"]), radius=5, color=(0, 0, 0), thickness=-1)
                plt.imshow(image)
                plt.show()
                plt.close()
                i+=1
                found=True
                break

    if(i>99):
        break

In [ ]:
#Codigo para crear blue_images.json

# l_blue_images=[]
# previous_baby_id=0
# for dic in data:
#     baby_id=extract_baby_id(dic)

#     if(baby_id!=previous_baby_id):
#         print(baby_id)

#     try:
#         image=np.array(cv2.cvtColor(cv2.imread(os.path.join(BASE_PATH, dic["aligned_image"])), cv2.COLOR_BGR2RGB))
#     except Exception as e:
#         print(e, "Error with path:", os.path.join(BASE_PATH, dic["aligned_image"]))

#     canal_maximo=np.argmax(image, axis=-1)

#     azules=(canal_maximo==2)

#     porcentaje_azul=np.mean(azules)
    
#     if porcentaje_azul >= 0.80:
#         l_blue_images.append(os.path.join(BASE_PATH, dic["aligned_image"]))
    
#     previous_baby_id=baby_id

#Comprobacion como funciona lo de los outliers por beb

# i=0
# for dic in l_babies[23]:
#     baby_id=extract_baby_id(dic)

#     if not color:
#         if(i<10):
#             print(baby_id)
#             new_color=color_extract(dic)
#             if(new_color):
#                 i+=1

#Mostrar imagenes eliminadas en el extract_color

# for point in dic["anotaciones"]:
        #     cv2.circle(image, (point["x"], point["y"]), radius=5, color=(0, 0, 0), thickness=-1)
        
        # plt.imshow(cv2.cvtColor(image, cv2.COLOR_HSV2RGB))
        # plt.show()
        # plt.close()

        # inverse_mask=np.array([not x for x in list(mask)])
        # eliminated=pixels[inverse_mask]

        # if(len(list(eliminated))!=0):
        #     for point in list(eliminated):
        #         print(point)
        #         im_color=np.full((200, 200, 3), point, dtype=np.uint8)
        #         plt.imshow(cv2.cvtColor(im_color, cv2.COLOR_HSV2RGB))
        #         plt.title("Eliminated")
        #         plt.show()
        #         plt.close()